In [12]:
import numpy as np
import pymc as pm

R = 2.0
N = 40
r = 0.1
a = 0.5
Y = 10000

true_energy = 1.2  # МэВ
true_pos = np.array([0.0, 0.0, 0.3])  

phi = np.pi * (np.sqrt(5.0) - 1.0)
pmt_positions = []
for i in range(N):
    y = 1.0 - (i / float(N - 1)) * 2.0
    rad_y = np.sqrt(1.0 - y * y)
    theta = phi * i
    pmt_positions.append(np.array([np.cos(theta) * rad_y, y, np.sin(theta) * rad_y]) * R)
pmt_positions = np.array(pmt_positions)
pmt_angular_radius = np.arcsin(r / R)

n_true_photons = np.random.poisson(true_energy * Y)
phi_arr = np.random.uniform(0, 2 * np.pi, n_true_photons)
costheta_arr = np.random.uniform(-1.0, 1.0, n_true_photons)
sintheta_arr = np.sqrt(1.0 - costheta_arr**2)
directions = np.column_stack((sintheta_arr * np.cos(phi_arr), sintheta_arr * np.sin(phi_arr), costheta_arr))
r0 = np.tile(true_pos, (n_true_photons, 1))

# точки пересечения
b = 2 * np.sum(r0 * directions, axis=1)
c = np.sum(r0**2, axis=1) - R**2
t = (-b + np.sqrt(b**2 - 4 * c)) / 2.0
hit_points = r0 + directions * t[:, np.newaxis]

# проверка попаданий на каждый ФЭУ с учетом затухания
norm_hits = hit_points / np.linalg.norm(hit_points, axis=1, keepdims=True)
norm_pmts = pmt_positions / R
cos_angles = np.clip(np.dot(norm_hits, norm_pmts.T), -1.0, 1.0)
angles = np.arccos(cos_angles)
hits_mask = angles <= pmt_angular_radius
hit_indices = np.argmax(hits_mask, axis=1)
has_hit = np.any(hits_mask, axis=1)
hit_indices[~has_hit] = -1
valid_hits = hit_indices[hit_indices != -1]
pmt_hits_true = np.bincount(valid_hits, minlength=N)

# наблюдаемый отклик фотокатодов 
observed_pe = np.random.poisson(a * pmt_hits_true)

print(f"Истинная энергия: {true_energy} МэВ | Истинная позиция: {true_pos}")
print(f"Суммарный зарегистрированный заряд: {np.sum(observed_pe)} ф.э.")


with pm.Model() as reconstruction_model:
    energy_rec = pm.Uniform('energy', lower=0.1, upper=3.0)
    z_pos_rec = pm.Normal('z_pos', mu=0.0, sigma=0.5)  
    
    # ожидаемое число фотонов
    mu_photons = energy_rec * Y
    
    # зададим аппроксимацию ожидаемого отклика для каждого ФЭУ с учетом z-смещения
    z_weights = 1.0 + (pmt_positions[:, 2] / R) * z_pos_rec
    z_weights = pm.math.clip(z_weights, 0.1, 2.0)
    
    # ожидаемое число фотоэлектронов на каждом ФЭУ
    base_prob = (np.pi * r**2) / (4 * np.pi * R**2) * a
    expected_pe_per_pmt = mu_photons * (base_prob * z_weights)
    
    pm.Poisson('obs_pe', mu=expected_pe_per_pmt, observed=observed_pe)
    
    with reconstruction_model:
        trace = pm.sample(draws=1000, tune=1000, return_inferencedata=True, random_seed=42)

reconstructed_energy = trace.posterior['energy'].mean().values
reconstructed_z = trace.posterior['z_pos'].mean().values

print(f"Реконструированная энергия: {reconstructed_energy:.2f} МэВ (истинная: {true_energy})")
print(f"Реконструированная позиция z: {reconstructed_z:.2f} м (истинная: {true_pos[2]})")

Истинная энергия: 1.2 МэВ | Истинная позиция: [0.  0.  0.3]
Суммарный зарегистрированный заряд: 136 ф.э.


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [energy, z_pos]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 14 seconds.


Реконструированная энергия: 1.10 МэВ (истинная: 1.2)
Реконструированная позиция z: 0.25 м (истинная: 0.3)
